In [ ]:
import pandas as pd
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier
#write your code here

true = pd.read_csv("https://confrecordings.ams3.digitaloceanspaces.com/1_True1.csv")

fake = pd.read_csv("https://confrecordings.ams3.digitaloceanspaces.com/Fake1.csv")

print(true.head())
print(fake.head())

true['target'] = 'true'
fake['target'] = 'fake'

data = pd.concat([fake, true]).reset_index(drop = True)

data = shuffle(data)
data = data.reset_index(drop=True)

data.drop(["date"],axis=1,inplace=True)

data.drop(["title"],axis=1,inplace=True)

data.drop(["subject"],axis=1,inplace=True)

print(data.head())



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


                                               title  \
0  As U.S. budget fight looms, Republicans flip t...   
1  U.S. military to accept transgender recruits o...   
2  Senior U.S. Republican senator: 'Let Mr. Muell...   
3  FBI Russia probe helped by Australian diplomat...   
4  Trump wants Postal Service to charge 'much mor...   

                                                text       subject  \
0  WASHINGTON (Reuters) - The head of a conservat...  politicsNews   
1  WASHINGTON (Reuters) - Transgender people will...  politicsNews   
2  WASHINGTON (Reuters) - The special counsel inv...  politicsNews   
3  WASHINGTON (Reuters) - Trump campaign adviser ...  politicsNews   
4  SEATTLE/WASHINGTON (Reuters) - President Donal...  politicsNews   

                 date  
0  December 31, 2017   
1  December 29, 2017   
2  December 31, 2017   
3  December 30, 2017   
4  December 29, 2017   
                                               title  \
0   Donald Trump Sends Out Embarrassing Ne

In [ ]:
wordnet = WordNetLemmatizer()
corpus = []

#texts = data['text'].values
def clean_text(text):
    # Cleaning text
    review = re.sub('[^a-zA-Z]', ' ', text)
    # Formating text
    review = review.lower()
    review = review.split()
    # Lemmatize each word in text
    review = [wordnet.lemmatize(word)
              for word in review
              if word not in stopwords.words('english')]
    review = ' '.join(review)
    return review

data["text"] = data["text"].apply(clean_text)
print(data.head())

X = data["text"]

y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=42)

tfidf_v = TfidfVectorizer()
X_tfidf = tfidf_v.fit_transform(X_train)
X1_tfidf = tfidf_v.transform(X_test)



                                                text target
0  washington reuters u president donald trump co...   true
1  wake yet another court decision derailed donal...   fake
2  following statement posted verified twitter ac...   true
3  pope francis used annual christmas day message...   fake
4  washington reuters trump campaign adviser geor...   true


In [ ]:
classifier1 = LogisticRegression(random_state = 0)
classifier1.fit(X_tfidf, y_train)
prediction = classifier1.predict(X1_tfidf) # Predict on new data
logacc = "accuracy: {}%".format(round(accuracy_score(y_test, prediction)*100,2))
print(logacc)
cm = metrics.confusion_matrix(y_test, prediction)
print(cm)


accuracy: 85.0%
[[ 6  0]
 [ 3 11]]


In [ ]:
tree = DecisionTreeClassifier()
tree.fit(X_tfidf, y_train)
predictiontree = tree.predict(X1_tfidf)
treeacc = "accuracy: {}%".format(round(accuracy_score(y_test, predictiontree)*100,2))
print(treeacc)
cm1 = metrics.confusion_matrix(y_test, predictiontree)
print(cm1)

accuracy: 100.0%
[[ 6  0]
 [ 0 14]]


In [ ]:
rdc = RandomForestClassifier()
rdc.fit(X_tfidf, y_train)
predictionforest = rdc.predict(X1_tfidf)
foracc = "accuracy: {}%".format(round(accuracy_score(y_test, predictionforest)*100,2))
print(foracc)
cm2 = metrics.confusion_matrix(y_test, predictionforest)
print(cm2)

accuracy: 100.0%
[[ 6  0]
 [ 0 14]]


In [ ]:
KNN = KNeighborsClassifier()
KNN.fit(X_tfidf, y_train)
predictionKNN = KNN.predict(X1_tfidf)
knnacc = "accuracy: {}%".format(round(accuracy_score(y_test, predictionKNN)*100,2))
print(knnacc)
cm3 = metrics.confusion_matrix(y_test, predictionKNN)
print(cm3)

accuracy: 75.0%
[[ 2  4]
 [ 1 13]]


In [ ]:
def prediction(news):
    new_xv_test = tfidf_v.transform([news])
    pred_LR = rdc.predict(new_xv_test)
    print(pred_LR)
prediction('House Intelligence Committee Chairman Devin Nunes is going to have a bad day. He s been under the assumption, like many of us, that the Christopher Steele-dossier was what prompted the Russia investigation so he s been lashing out at the Department of Justice and the FBI in order to protect Trump. As it happens, the dossier is not what started the investigation, according to documents obtained by the New York Times.Former Trump campaign adviser George Papadopoulos was drunk in a wine bar when he revealed knowledge of Russian opposition research on Hillary Clinton.On top of that, Papadopoulos wasn t just a covfefe boy for Trump, as his administration has alleged. He had a much larger role, but none so damning as being a drunken fool in a wine bar. Coffee boys  don t help to arrange a New York meeting between Trump and President Abdel Fattah el-Sisi of Egypt two months before the election. It was known before that the former aide set up meetings with world leaders for Trump, but team Trump ran with him being merely a coffee boy.In May 2016, Papadopoulos revealed to Australian diplomat Alexander Downer that Russian officials were shopping around possible dirt on then-Democratic presidential nominee Hillary Clinton. Exactly how much Mr. Papadopoulos said that night at the Kensington Wine Rooms with the Australian, Alexander Downer, is unclear,  the report states.  But two months later, when leaked Democratic emails began appearing online, Australian officials passed the information about Mr. Papadopoulos to their American counterparts, according to four current and former American and foreign officials with direct knowledge of the Australians  role. Papadopoulos pleaded guilty to lying to the F.B.I. and is now a cooperating witness with Special Counsel Robert Mueller s team.This isn t a presidency. It s a badly scripted reality TV show.Photo by Win McNamee/Getty Images.')
prediction('WASHINGTON (Reuters) - Senate Majority Leader Mitch McConnell said on Monday he was confident the U.S. Congress would be able to reach an agreement to fund the government when the current spending bill ends on Dec. 22 and that there would be no forced government shutdown. â€œThere isnâ€™t any chance we are going to shut the government down. Weâ€™re in discussions, not only on a cap deal, but also on the way forward on appropriations, McConnell told reporters. â€œThe American people need not worry that there is going to be any kind of government shutdown.â€ But U.S. Senate Democratic leader Chuck Schumer said a full-year defense funding bill with short-term money for other programs would fail in the Senate. â€œDemocrats will oppose any budget deal that would allow defense spending to increase while holding down domestic priorities,â€ he said to reporters.  ')


['fake']
['true']
